# Diffusion style transfer: demo of the fixed `diffuser/` package

This notebook used to contain a hand-rolled, single-image-overfit DDPM prototype directly in its cells. That prototype performed terribly: the noise-prediction MSE loss trended *up* over training instead of down, the perceptual loss never converged, and the sampled output was a dark, noise-dominated, red-blown-out mess.

The root causes (perceptual loss scored at every timestep instead of just low-noise ones, unclamped intermediate sampling steps, a too-shallow UNet with no attention and a dropped skip connection, a linear rather than cosine noise schedule, no EMA/gradient clipping, and single-image-only training) are now fixed in the `diffuser/` package, which:

- Uses `diffusers`' `UNet2DModel` (proper attention, full skip connections) instead of a hand-rolled tiny UNet.
- Uses a cosine beta schedule and every-step-clamped DDIM sampling instead of a hand-rolled linear schedule with only a final clamp.
- Restricts `perceptual_loss` to low-noise timesteps via a configurable cutoff.
- Trains with EMA and gradient clipping on a real (if small, for this demo) multi-image dataset via `DataLoader`s.

This notebook is now just a thin demo: it trains a tiny dry-run config from `diffuser.config.DiffusionExperiments` for a couple of minutes and samples from it, as an end-to-end smoke check of the package -- not a from-scratch implementation. See `diffuser/` for the actual model/training/inference code and `tests/test_diffuser_*.py` for the regression guards on each fix above.

In [ ]:
import os
import sys
from pathlib import Path

# walk up from the notebook's cwd until we find the repo root (style_transfer/)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "style_transfer").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# diffuser.config's dataset paths (and train_model's models/<name> checkpoint
# path, and find_checkpoint's lookup) are all plain relative strings, meant
# to be run from the repo root -- same as run_experiment.py /
# run_diffusion_experiment.py. Jupyter's kernel cwd is the notebook's own
# directory, not the repo root, so without this chdir those relative paths
# resolve against notebooks/ instead and raise "No images found".
os.chdir(REPO_ROOT)

from PIL import Image

from style_transfer.inference import select_device, postprocess, find_checkpoint
from style_transfer.dataset import SingleImageDataset

from diffuser.config import DiffusionExperiments
from diffuser.train import train_model
from diffuser.inference import load_model, sample

DEVICE = select_device()
print(f"Using device: {DEVICE}")

## Config

`flowers_kanagawa_dry_run` is a small/fast config (32px images, 1 epoch, 10% of the flowers dataset) meant for exactly this kind of quick smoke check. `flowers_kanagawa` (128px, full dataset, ~20 epochs) is the real experiment -- run it via `run_diffusion_experiment.py` at the repo root, not in this notebook.

In [ ]:
EXPERIMENT_NAME = "diffusion_notebook_demo"
cfg = DiffusionExperiments["flowers_kanagawa_dry_run"]
cfg

## Train

`train_model` mirrors `style_transfer.train.train_model`'s shape: it initializes the VGG feature extractor, builds the `ConditionedUNet` + EMA + optimizer + scheduler, trains on real `DataLoader`-batched content/style images, checkpoints each epoch, and copies EMA weights into the model before the final save.

In [ ]:
train_model(EXPERIMENT_NAME, cfg, DEVICE)

## Sample + compare

Load the trained checkpoint and run the fixed reverse-diffusion sampling loop (`diffuser.inference.sample`), which clamps every intermediate step via the scheduler rather than only the final output.

In [ ]:
def side_by_side(*tensors):
    imgs = [postprocess(t.squeeze(0).cpu()) for t in tensors]
    widths, heights = zip(*(im.size for im in imgs))
    canvas = Image.new("RGB", (sum(widths), max(heights)))
    x = 0
    for im in imgs:
        canvas.paste(im, (x, 0))
        x += im.width
    return canvas

CONTENT_PATH = REPO_ROOT / "artifacts/images/content/flowers/lily6.jpeg"

content_tensor = SingleImageDataset(str(CONTENT_PATH), cfg.image_size, DEVICE, quiet=True)[0].unsqueeze(0).to(DEVICE)
style_tensor = SingleImageDataset(cfg.style["dataset"], cfg.image_size, DEVICE, quiet=True)[0].unsqueeze(0).to(DEVICE)

model_path = find_checkpoint(EXPERIMENT_NAME)
model = load_model(model_path, cfg, DEVICE)

styled_tensor = sample(model, content_tensor, cfg, DEVICE)

side_by_side(content_tensor, style_tensor, styled_tensor)

## Save

In [ ]:
OUTPUT_DIR = REPO_ROOT / "outputs/quick_test/diffusion_experiment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUTPUT_DIR / "lily6_kanagawa_diffuser_demo.jpg"
postprocess(styled_tensor.squeeze(0).cpu()).save(out_path)
print(f"Saved styled image to {out_path}")

## Notes

This dry-run config trains for one epoch on 10% of the flowers dataset at 32px -- enough to confirm the pipeline runs end to end and produces a bounded, non-degenerate image, not enough to expect strong stylization. For an actual trained model, run the real `flowers_kanagawa` experiment (128px, full dataset, ~20 epochs) via `run_diffusion_experiment.py`, then point this notebook's `EXPERIMENT_NAME`/`cfg` at it to sample from the result.

See `diffuser/` for the implementation (`schedule.py`, `models.py`, `loss.py`, `train.py`, `inference.py`, `config/`) and `tests/test_diffuser_*.py` for the automated regression guards on each of the root-cause fixes described at the top of this notebook.